In [ ]:
import json
import pandas as pd
from collections import defaultdict
import os

# Load your full sentiment CSV
df = pd.read_csv("stock_sentiment_news.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["sentiment"] = df["sentiment"].fillna("neutral")
df["date_str"] = df["date"].dt.strftime("%Y-%m-%d")

# Output structures
all_sentiment_data = {}
histogram = defaultdict(int)
timeline = defaultdict(int)

# Loop through each stock
for symbol in df["symbol"].unique():
    df_stock = df[df["symbol"] == symbol].copy()
    if df_stock.empty:
        continue

    total_articles = len(df_stock)
    sentiment_counts = df_stock["sentiment"].value_counts().to_dict()

    # Percent breakdown
    positive_percent = round((sentiment_counts.get("positive", 0) / total_articles) * 100, 2)
    neutral_percent = round((sentiment_counts.get("neutral", 0) / total_articles) * 100, 2)
    negative_percent = round((sentiment_counts.get("negative", 0) / total_articles) * 100, 2)

    # News list + chart prep
    articles = []
    bar_polarities = defaultdict(lambda: {"positive": 0, "neutral": 0, "negative": 0})
    line_labels = []
    line_scores = []

    for _, row in df_stock.iterrows():
        date_str = row["date_str"]
        sentiment = row["sentiment"]
        polarity = row["polarity"]

        articles.append({
            "title": row["title"],
            "link": row["link"],
            "date": date_str,
            "sentiment": sentiment,
            "polarity": polarity
        })

        # Bar polarity score (instead of count)
        if sentiment in ["positive", "neutral", "negative"]:
            bar_polarities[date_str][sentiment] = polarity

        # Line chart (all polarity scores)
        line_labels.append(date_str)
        line_scores.append(polarity)

        # Global sentiment breakdowns
        histogram[sentiment] += 1
        timeline[date_str] += 1

    # Top positive/negative
    df_sorted = df_stock.sort_values("polarity", ascending=False)
    top_positive = df_sorted.iloc[0][["title", "link", "polarity", "date_str"]].to_dict()
    top_negative = df_sorted.iloc[-1][["title", "link", "polarity", "date_str"]].to_dict()

    # Final bar chart data
    bar_labels = sorted(bar_polarities.keys())
    bar_data = {
        "positive": [bar_polarities[d]["positive"] for d in bar_labels],
        "neutral": [bar_polarities[d]["neutral"] for d in bar_labels],
        "negative": [bar_polarities[d]["negative"] for d in bar_labels],
    }

    # Add to master JSON
    all_sentiment_data[symbol] = {
        "symbol": symbol,
        "company": df_stock["company"].iloc[0],
        "total_articles": total_articles,
        "positive_percent": positive_percent,
        "neutral_percent": neutral_percent,
        "negative_percent": negative_percent,
        "top_positive_news": top_positive,
        "top_negative_news": top_negative,
        "articles": articles,
        "bar_labels": bar_labels,
        "bar_data": bar_data,
        "line_sentiment_labels": line_labels,
        "line_sentiment_scores": line_scores
    }

# Wrap everything together
combined_sentiment_json = {
    "stocks": all_sentiment_data,
    "charts": {
        "histogram": dict(histogram),
        "timeline": {
            "dates": list(timeline.keys()),
            "counts": list(timeline.values())
        }
    }
}

# Save to one JSON file
os.makedirs("sentiment_data", exist_ok=True)
with open("sentiment_data/sentiment_all_stocks.json", "w", encoding="utf-8") as f:
    json.dump(combined_sentiment_json, f, ensure_ascii=False, indent=2)

print("✅ All-in-one sentiment JSON enriched and saved: sentiment_data/sentiment_all_stocks.json")


✅ All-in-one sentiment JSON enriched and saved: sentiment_data/sentiment_all_stocks.json
